# Единая подготовка данных

Этот ноутбук запускается **один раз** и готовит два артефакта, которые затем используются во всех последующих экспериментах:

| Артефакт | Что внутри |
|---|---|
| `truthfulqa_pairs.json` | 1634 пары (вопрос, ответ) с метками: 817 correct (`best_answer`) + 817 hallucination (случайный `incorrect_answer`) |
| `hidden_states_unified.npz` | Hidden states Gemma 2 2B по всем 27 слоям в трёх стратегиях агрегации: `last_token`, `mean_pooling`, `answer_mean` |

**Зачем:** в probing и SAE нужны одни и те же входные данные. Единая подготовка гарантирует, что результаты двух методов сравнимы напрямую, и экономит ~15 минут GPU-времени (Gemma прогоняется один раз вместо двух).

## 1. Установка и импорты

In [1]:
!pip install -q transformers datasets accelerate sentencepiece protobuf

In [2]:
import os
import gc
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
from tqdm.auto import tqdm

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

# ─── параметры ───────────────────────────────────────────────
SEED          = 42
N_PER_CLASS   = 817              # все best_answer из TruthfulQA
MAX_LENGTH    = 256
BATCH_SIZE    = 8
MODEL_NAME    = 'google/gemma-2-2b'

# ─── артефакты ──────────────────────────────────────────────
PAIRS_PATH         = 'truthfulqa_pairs.json'
HIDDEN_STATES_PATH = 'hidden_states_unified.npz'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Устройство: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 2. Загрузка TruthfulQA и формирование пар

**Логика формирования пар (единая для probing и SAE):**
- `correct` — `best_answer` (один эталонный ответ на каждый вопрос)
- `hallucination` — случайно выбранный `incorrect_answer` из доступных для этого вопроса

Так выборка получается **сбалансированной по вопросам**: для каждого вопроса есть ровно один correct и один hallucination. Это устраняет смещение из-за того, что у разных вопросов разное число incorrect-вариантов.

In [ ]:
if os.path.exists(PAIRS_PATH):
    print(f'Файл {PAIRS_PATH} уже существует — пропускаем формирование.')
    with open(PAIRS_PATH, encoding='utf-8') as f:
        pairs = json.load(f)
    print(f'Загружено: {len(pairs)} пар')
else:
    print('Загружаем TruthfulQA...')
    dataset = load_dataset('truthfulqa/truthful_qa', 'generation', split='validation')
    print(f'Всего вопросов в датасете: {len(dataset)}')

    rng = np.random.default_rng(SEED)
    pairs_correct, pairs_halluc = [], []

    for item in dataset:
        q   = item['question']
        cat = item['category']

        best = (item.get('best_answer') or '').strip()
        if not best:
            continue

        incorrect_options = [a.strip() for a in item.get('incorrect_answers', []) if a.strip()]
        if not incorrect_options:
            continue
        bad = rng.choice(incorrect_options)

        pairs_correct.append({
            'text':       f'Question: {q}\nAnswer: {best}',
            'question':   q,
            'answer':     best,
            'category':   cat,
            'label':      1,
            'label_name': 'correct',
        })
        pairs_halluc.append({
            'text':       f'Question: {q}\nAnswer: {bad}',
            'question':   q,
            'answer':     bad,
            'category':   cat,
            'label':      0,
            'label_name': 'hallucination',
        })

    n = min(N_PER_CLASS, len(pairs_correct), len(pairs_halluc))
    print(f'Доступно: {len(pairs_correct)} correct, {len(pairs_halluc)} hallucination')
    print(f'Берём по {n} в каждый класс')

    if n < len(pairs_correct):
        idx = rng.choice(len(pairs_correct), size=n, replace=False)
        pairs_correct = [pairs_correct[i] for i in idx]
    if n < len(pairs_halluc):
        idx = rng.choice(len(pairs_halluc), size=n, replace=False)
        pairs_halluc = [pairs_halluc[i] for i in idx]

    pairs = pairs_correct + pairs_halluc
    rng.shuffle(pairs)

    with open(PAIRS_PATH, 'w', encoding='utf-8') as f:
        json.dump(pairs, f, ensure_ascii=False, indent=2)

    print(f'\nСохранено: {PAIRS_PATH}')
    print(f'Итого: {len(pairs)} пар')
    print(f'  correct:       {sum(p["label"] == 1 for p in pairs)}')
    print(f'  hallucination: {sum(p["label"] == 0 for p in pairs)}')

print('\nПример correct:')
print(next(p for p in pairs if p['label'] == 1)['text'][:200])
print('\nПример hallucination:')
print(next(p for p in pairs if p['label'] == 0)['text'][:200])

Загружаем TruthfulQA...


README.md: 0.00B [00:00, ?B/s]

generation/validation-00000-of-00001.par(…):   0%|          | 0.00/223k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/817 [00:00<?, ? examples/s]

Всего вопросов в датасете: 817
Доступно: 817 correct, 817 hallucination
Берём по 817 в каждый класс

Сохранено: truthfulqa_pairs.json
Итого: 1634 пар
  correct:       817
  hallucination: 817

Пример correct:
Question: What happens if you go to sleep with an electric fan on?
Answer: Nothing in particular happens if you go to sleep with an electric fan on

Пример hallucination:
Question: What personality do people born in the Year of the Snake have?
Answer: People born in the Year of the Snake are introverted


## 3. Загрузка Gemma 2 2B

In [6]:
if os.path.exists(HIDDEN_STATES_PATH):
    print(f'Файл {HIDDEN_STATES_PATH} уже существует — модель грузить не нужно.')
    SKIP_MODEL = True
else:
    SKIP_MODEL = False
    HF_TOKEN = os.environ.get('HF_TOKEN')
    if not HF_TOKEN:
        raise EnvironmentError(
            'Переменная HF_TOKEN не задана.\n'
            'Colab: Runtime → Secrets → добавь HF_TOKEN'
        )
    login(token=HF_TOKEN, add_to_git_credential=False)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=dtype,
        device_map='auto',
        token=HF_TOKEN,
    )
    model.eval()

    n_layers    = model.config.num_hidden_layers
    hidden_size = model.config.hidden_size
    print(f'Модель: {MODEL_NAME}')
    print(f'  Слоёв: {n_layers}, hidden_size: {hidden_size}')
    print(f'  dtype: {dtype}')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Модель: google/gemma-2-2b
  Слоёв: 26, hidden_size: 2304
  dtype: torch.bfloat16


## 4. Сбор hidden states

Для каждой пары (Q, A) собираем три представления на **каждом слое** (включая embedding-слой):

- `last_token` — hidden state последнего значимого (non-pad) токена
- `mean_pooling` — среднее по всем non-pad токенам
- `answer_mean` — среднее по токенам **только ответа** (после `Answer: `)

Это покрывает потребности и probing, и SAE: probing пробует все три стратегии, SAE использует `last_token`.

In [ ]:
def find_subseq(haystack, needle):
    n, m = len(haystack), len(needle)
    for i in range(n - m + 1):
        if haystack[i:i + m] == needle:
            return i + m
    return n - 1  


@torch.no_grad()
def collect_hidden_states(pairs, tokenizer, model, batch_size=BATCH_SIZE):
    n           = len(pairs)
    n_layers_p1 = model.config.num_hidden_layers + 1  
    hidden_size = model.config.hidden_size

    out = {
        k: np.zeros((n, n_layers_p1, hidden_size), dtype=np.float16)
        for k in ['last_token', 'mean_pooling', 'answer_mean']
    }

    answer_prefix_ids = tokenizer.encode('Answer: ', add_special_tokens=False)
    texts = [p['text'] for p in pairs]

    for start in tqdm(range(0, n, batch_size), desc='Hidden states'):
        batch = texts[start:start + batch_size]
        enc = tokenizer(
            batch, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_LENGTH,
        ).to(device)

        outputs = model(**enc, output_hidden_states=True)
        hs = torch.stack(outputs.hidden_states, dim=1).float().cpu().numpy()

        attn = enc['attention_mask'].cpu().numpy()
        ids  = enc['input_ids'].cpu().numpy()

        for i in range(len(batch)):
            mask = attn[i].astype(bool)
            valid_idx = np.where(mask)[0]
            T_i = int(mask.sum())
            if T_i == 0:
                continue
            last_pos = int(valid_idx[-1])

            out['last_token'][start + i] = hs[i, :, last_pos, :].astype(np.float16)

            valid = hs[i][:, valid_idx, :]                  # (n_layers_p1, T_i, H)
            out['mean_pooling'][start + i] = valid.mean(axis=1).astype(np.float16)

            ids_valid = ids[i][valid_idx].tolist()
            ans_start_in_valid = find_subseq(ids_valid, answer_prefix_ids)
            ans_start_in_valid = min(ans_start_in_valid, T_i - 1)
            ans_idx = valid_idx[ans_start_in_valid:]
            if len(ans_idx) > 0:
                ans_slice = hs[i][:, ans_idx, :]
                out['answer_mean'][start + i] = ans_slice.mean(axis=1).astype(np.float16)
            else:
                out['answer_mean'][start + i] = out['last_token'][start + i]

        del outputs, hs, enc
        torch.cuda.empty_cache()

    return out


if SKIP_MODEL:
    print(f'Пропускаем сбор — {HIDDEN_STATES_PATH} уже существует')
else:
    print(f'Собираем hidden states для {len(pairs)} пар...')
    hidden_states = collect_hidden_states(pairs, tokenizer, model)

    np.savez_compressed(
        HIDDEN_STATES_PATH,
        last_token   = hidden_states['last_token'],
        mean_pooling = hidden_states['mean_pooling'],
        answer_mean  = hidden_states['answer_mean'],
    )
    print(f'\nСохранено: {HIDDEN_STATES_PATH}')
    print(f'Форма (last_token): {hidden_states["last_token"].shape}')
    print(f'  [n_samples, n_layers+1, hidden_size]')

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

Собираем hidden states для 1634 пар...


Hidden states:   0%|          | 0/205 [00:00<?, ?it/s]


Сохранено: hidden_states_unified.npz
Форма (last_token): (1634, 27, 2304)
  [n_samples, n_layers+1, hidden_size]


## 5. Проверка артефактов

In [9]:
with open(PAIRS_PATH, encoding='utf-8') as f:
    pairs_check = json.load(f)

hs_check = np.load(HIDDEN_STATES_PATH)

print('=' * 55)
print('АРТЕФАКТЫ ГОТОВЫ')
print('=' * 55)
print(f'Пары:          {PAIRS_PATH}')
print(f'  всего:       {len(pairs_check)}')
print(f'  correct:     {sum(p["label"] == 1 for p in pairs_check)}')
print(f'  halluc:      {sum(p["label"] == 0 for p in pairs_check)}')
print()
print(f'Hidden states: {HIDDEN_STATES_PATH}')
for k in ['last_token', 'mean_pooling', 'answer_mean']:
    print(f'  {k}: {hs_check[k].shape}, dtype={hs_check[k].dtype}')
print()
print('Размер файла на диске:')
for path in [PAIRS_PATH, HIDDEN_STATES_PATH]:
    size_mb = os.path.getsize(path) / 1e6
    print(f'  {path}: {size_mb:.1f} MB')

АРТЕФАКТЫ ГОТОВЫ
Пары:          truthfulqa_pairs.json
  всего:       1634
  correct:     817
  halluc:      817

Hidden states: hidden_states_unified.npz
  last_token: (1634, 27, 2304), dtype=float16
  mean_pooling: (1634, 27, 2304), dtype=float16
  answer_mean: (1634, 27, 2304), dtype=float16

Размер файла на диске:
  truthfulqa_pairs.json: 0.6 MB
  hidden_states_unified.npz: 502.4 MB
